# Module 22 · Ước lượng tham số

**Nguồn sách:** Barkat, chương 6, tr. 345 đến 390

Notebook đi kèm bài giảng cùng số module. Chạy lần lượt từng cell; sau mỗi cell có phần **📤 Đầu ra thật** giải thích con số.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (7.5, 3.3), "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 110})


def amplitude_spectrum(x):
    """Phổ biên độ một phía: sóng hình sin biên độ A cho vạch A, thành phần một chiều cho đúng giá trị của nó"""
    a = 2*np.abs(np.fft.rfft(x))/len(x)
    a[0] /= 2
    return a


def report(key, value, fmt=".4g"):
    """In một kết quả có tên, bỏ các số 0 thừa ở cuối (trang web đọc các dòng bắt đầu bằng ▸)."""
    s = f"{value:{fmt}}"
    if "." in s and "e" not in s:
        s = s.rstrip("0").rstrip(".")
    print(f"▸ {key} = {s}")


## 1. Ước lượng ML: trung bình, phương sai, và phương trình hợp lý
🎯 **Phương pháp này trả lời câu hỏi gì?** Ước lượng ML của trung bình Gauss có khớp công thức đóng khi dò tìm bằng lưới số, đạo hàm hàm hợp lý tại điểm đó có bằng 0, và ước lượng ML của phương sai có khớp công thức đóng khi $m$ đã biết?

In [2]:
import numpy as np
from scipy import stats, integrate, optimize
rg = np.random.default_rng(22)

m_true, sigma_true, K = 3.0, 2.0, 8
y_sample = rg.normal(m_true, sigma_true, K)

def loglik_m(m_, ys, sig): return np.sum(stats.norm.logpdf(ys, m_, sig))
m_grid = np.linspace(-2, 8, 20001)
ll = np.array([loglik_m(mv, y_sample, sigma_true) for mv in m_grid])
ml_grid_est = m_grid[np.argmax(ll)]
ml_closed_est = y_sample.mean()
report("ml_grid_est", ml_grid_est, ".4f")
report("ml_closed_est", ml_closed_est, ".4f")
assert abs(ml_grid_est - ml_closed_est) < 1e-3

h = 1e-5
deriv = (loglik_m(ml_closed_est+h, y_sample, sigma_true) - loglik_m(ml_closed_est-h, y_sample, sigma_true))/(2*h)
report("ml_deriv_zero", deriv, ".2e")
assert abs(deriv) < 1e-6

ml_var_est = np.mean((y_sample - m_true)**2)
report("ml_var_est", ml_var_est, ".4f")
report("ml_var_true", sigma_true**2, ".4f")

ls_vals = np.array([np.sum((y_sample-mv)**2) for mv in m_grid])
ls_grid_est = m_grid[np.argmin(ls_vals)]
report("ls_grid_est", ls_grid_est, ".4f")
assert abs(ls_grid_est - ml_closed_est) < 1e-3

▸ ml_grid_est = 2.3815
▸ ml_closed_est = 2.3815
▸ ml_deriv_zero = 0.00e+00
▸ ml_var_est = 6.3152
▸ ml_var_true = 4
▸ ls_grid_est = 2.3815


#### 📤 Đầu ra thật
Ước lượng ML dò lưới 2.3815 khớp công thức đóng 2.3815; đạo hàm tại điểm đó 0.00e+00; phương sai ML 6.3152 so với thật 4; bình phương tối thiểu dò lưới 2.3815 khớp cùng điểm.

## 2. Độ chệch: Monte Carlo cho $\hat m_{ml}$ và $\hat\sigma^2_{ml}$
🎯 **Phương pháp này trả lời câu hỏi gì?** Trung bình thực nghiệm của $\hat m_{ml}$ và $\hat\sigma^2_{ml}$ (khi $m$ đã biết) qua nhiều lần thử có khớp giá trị thật (không chệch), và $\hat\sigma^2_{ml}$ có thực sự trở nên chệch đúng theo hệ số $(K-1)/K$ khi $m$ cũng phải ước lượng?

In [3]:
n_trials = 20000
m_hats = np.empty(n_trials)
s2_hats_known_m = np.empty(n_trials)
s2_hats_est_m = np.empty(n_trials)
for i in range(n_trials):
    y = rg.normal(m_true, sigma_true, K)
    mh = y.mean()
    m_hats[i] = mh
    s2_hats_known_m[i] = np.mean((y - m_true)**2)
    s2_hats_est_m[i] = np.mean((y - mh)**2)

report("ex3_mean_m", m_hats.mean(), ".4f")
report("ex3_mean_s2_known_m", s2_hats_known_m.mean(), ".4f")
report("ex3_mean_s2_est_m", s2_hats_est_m.mean(), ".4f")
biased_pred = (K-1)/K*sigma_true**2
report("ex3_biased_pred", biased_pred, ".4f")
assert abs(m_hats.mean() - m_true) < 0.02
assert abs(s2_hats_known_m.mean() - sigma_true**2) < 0.05
assert abs(s2_hats_est_m.mean() - biased_pred) < 0.05

Ks = [4, 16, 64]
variances = []
for Kc in Ks:
    trials_c = np.array([rg.normal(m_true, sigma_true, Kc).mean() for _ in range(8000)])
    variances.append(trials_c.var())
report("cons_var4", variances[0], ".4f")
report("cons_var16", variances[1], ".4f")
report("cons_var64", variances[2], ".4f")
assert variances[0] > variances[1] > variances[2]

▸ ex3_mean_m = 3.0086
▸ ex3_mean_s2_known_m = 4.0016
▸ ex3_mean_s2_est_m = 3.4997
▸ ex3_biased_pred = 3.5


▸ cons_var4 = 1.0151
▸ cons_var16 = 0.2433
▸ cons_var64 = 0.0641


#### 📤 Đầu ra thật
Trung bình $\hat m_{ml}$: 3.0086; $\hat\sigma^2_{ml}$ (m biết): 4.0016; $\hat\sigma^2_{ml}$ (m ước lượng): 3.4997 so với dự đoán chệch 3.5; phương sai theo K: 1.0151, 0.2433, 0.0641.

## 3. Ước lượng Bayes: MMSE bằng số, và Ví dụ 6.5 rời rạc
🎯 **Phương pháp này trả lời câu hỏi gì?** Trung bình có điều kiện tính bằng tích phân số có khớp công thức đóng $\frac1{K+1}\sum y_k$ của Ví dụ 6.4, và $\hat x_{map}$, $\hat x_{ms}$ của Ví dụ 6.5 có đúng hành vi nhảy bậc và hằng số như dự đoán?

In [4]:
sigma_ex4 = 1.0; sigma_m_ex4 = sigma_ex4; K4 = 5
y4 = rg.normal(1.0, sigma_ex4, K4)

def num_integrand(m_, mult):
    return (m_**mult)*np.exp(-0.5*(np.sum((y4-m_)**2) + m_**2/sigma_m_ex4**2)/sigma_ex4**2 * sigma_ex4**2/sigma_ex4**2) if False else 0
# direct definition matching the book's exponent
def unnorm(m_):
    return np.exp(-0.5*(np.sum((y4-m_)**2)/sigma_ex4**2 + m_**2/sigma_m_ex4**2))
num,_ = integrate.quad(lambda m_: m_*unnorm(m_), -30, 30)
den,_ = integrate.quad(unnorm, -30, 30)
ex4_num_mean = num/den
ex4_closed_mean = np.sum(y4)/(K4+1)
report("ex4_num_mean", ex4_num_mean, ".4f")
report("ex4_closed_mean", ex4_closed_mean, ".4f")
assert abs(ex4_num_mean - ex4_closed_mean) < 1e-6

def x_map(n_): return 1.0 if n_ >= 0.5 else 0.0
def x_ms(n_): return np.exp(-n_-1)/(np.exp(-n_)+np.exp(-n_-1))
ns = [-1.0, 0.0, 0.5, 1.0, 2.0]
map_seq = [x_map(n_) for n_ in ns]
ms_seq = [x_ms(n_) for n_ in ns]
report("ex5_map_seq", ",".join(str(int(v)) for v in map_seq), "s")
report("ex5_ms_const", ms_seq[0], ".4f")
report("ex5_ms_seq", ",".join("%.4f" % v for v in ms_seq), "s")
assert map_seq == [0.0, 0.0, 1.0, 1.0, 1.0]
assert max(ms_seq) - min(ms_seq) < 1e-9
assert abs(ms_seq[0] - 1/(np.e+1)) < 1e-9

▸ ex4_num_mean = 0.6059
▸ ex4_closed_mean = 0.6059
▸ ex5_map_seq = 0,0,1,1,1
▸ ex5_ms_const = 0.2689
▸ ex5_ms_seq = 0.2689,0.2689,0.2689,0.2689,0.2689


#### 📤 Đầu ra thật
Trung bình MMSE số 0.6059 khớp công thức đóng 0.6059; dãy MAP 0,0,1,1,1; hằng số MMSE 0.2689, dãy đầy đủ 0.2689,0.2689,0.2689,0.2689,0.2689.

## 4. Cramér–Rao: đạt cận bằng mô phỏng
🎯 **Phương pháp này trả lời câu hỏi gì?** Phương sai thực nghiệm của $\hat m_{ml}$ qua nhiều lần thử có gần như trùng khít cận Cramér–Rao lý thuyết $\sigma^2/K$, xác nhận đây là một ước lượng hiệu quả?

In [5]:
sigma_cr = 1.5; K_cr = 10; trials_cr = 50000
mh_cr = rg.normal(3.0, sigma_cr, size=(trials_cr, K_cr)).mean(axis=1)
cr_var_empirical = mh_cr.var()
cr_var_bound = sigma_cr**2/K_cr
report("cr_var_empirical", cr_var_empirical, ".4f")
report("cr_var_bound", cr_var_bound, ".4f")
assert abs(cr_var_empirical - cr_var_bound)/cr_var_bound < 0.03

▸ cr_var_empirical = 0.2249
▸ cr_var_bound = 0.225


#### 📤 Đầu ra thật
Phương sai thực nghiệm 0.2249 so với cận Cramér–Rao 0.225.

## 5. BLUE và bình phương tối thiểu
🎯 **Phương pháp này trả lời câu hỏi gì?** Hệ số $(a,b)$ của BLUE tính từ công thức đóng có thực sự cực tiểu rủi ro bình phương trung bình so với các điểm lân cận, và nghiệm bình phương tối thiểu đóng $(H^TH)^{-1}H^TY$ có khớp giải số trực tiếp?

In [6]:
n_blue = 200000
theta_b = rg.normal(2.0, 1.5, n_blue)
Y_b = 0.7*theta_b + rg.normal(0.0, 1.0, n_blue)
Etheta = theta_b.mean(); EY = Y_b.mean()
EthetaY = (theta_b*Y_b).mean(); EY2 = (Y_b**2).mean()
a_blue = (EthetaY - Etheta*EY)/(EY2 - EY**2)
b_blue = Etheta - EY*a_blue
report("blue_a", a_blue, ".4f")
report("blue_b", b_blue, ".4f")

def mse_blue(a_, b_): return np.mean((theta_b - (a_*Y_b + b_))**2)
mse_opt = mse_blue(a_blue, b_blue)
report("blue_mse_opt", mse_opt, ".4f")
margins = []
for da in (-0.01, 0.01):
    for db in (-0.01, 0.01):
        margins.append(mse_blue(a_blue+da, b_blue+db) - mse_opt)
report("blue_mse_margin", min(margins), ".2e")
assert min(margins) > -1e-6

K_ls, M_ls = 6, 2
H_ls = rg.normal(size=(K_ls, M_ls))
theta_true_ls = np.array([1.5, -2.0])
N_ls = rg.normal(0, 0.3, K_ls)
Y_ls = H_ls @ theta_true_ls + N_ls
theta_closed = np.linalg.inv(H_ls.T @ H_ls) @ H_ls.T @ Y_ls
theta_numeric, *_ = np.linalg.lstsq(H_ls, Y_ls, rcond=None)
report("lse_closed", ",".join("%.4f" % v for v in theta_closed), "s")
report("lse_numeric", ",".join("%.4f" % v for v in theta_numeric), "s")
report("lse_dev", np.max(np.abs(theta_closed - theta_numeric)), ".2e")
assert np.max(np.abs(theta_closed - theta_numeric)) < 1e-9

▸ blue_a = 0.7508
▸ blue_b = 0.9472
▸ blue_mse_opt = 1.0735
▸ blue_mse_margin = 2.25e-04
▸ lse_closed = 1.3883,-1.7178
▸ lse_numeric = 1.3883,-1.7178
▸ lse_dev = 8.88e-16


#### 📤 Đầu ra thật
BLUE: $(a,b)$=(0.7508, 0.9472), rủi ro 1.0735, chênh nhỏ nhất 2.25e-04; bình phương tối thiểu: đóng 1.3883,-1.7178 so với số 1.3883,-1.7178 (lệch 8.88e-16).